# Direct S3 Benchmark — Bypass climakitae for Data Retrieval

**Hypothesis:** climakitae's `get_data()` adds ~29s overhead per call (DataParameters init,
boundary loading, catalog search) and its hardcoded `storage_options` prevent us from
using fsspec caching. By going directly to the S3 Zarr stores, we can:

1. Eliminate the 29s metadata overhead
2. Use fsspec `simplecache` to cache Zarr chunks locally (instant on repeat runs)
3. Safely parallelize with threads (no DataParameters thread-safety issue)

**Same test parameters as TestNotebook:**
- 1 region (Joshua Tree), 1 scenario (Historical), 5 years (2010-2014)
- 3 variables: T_Max, T_Min, Precip
- Full pipeline through cos-weighted spatial average
- Results saved to CSV for comparison with `test_output.csv`

## 1. Imports & Config

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray as rxr
import fsspec
import s3fs
import os
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

print(f"xarray:  {xr.__version__}")
print(f"fsspec:  {fsspec.__version__}")
print(f"s3fs:    {s3fs.__version__}")
print(f"numpy:   {np.__version__}")
print(f"pandas:  {pd.__version__}")
print()

# --- Paths ---
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
PARK_OUTLINES_DIR = os.path.join(PROJECT_ROOT, "parkOutlines")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "data", "csv")
CACHE_DIR = os.path.join(PROJECT_ROOT, "data", "zarr_cache")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Cache dir:    {CACHE_DIR}")
print(f"Output dir:   {OUTPUT_DIR}")

xarray:  2025.11.0
fsspec:  2026.1.0
s3fs:    2026.1.0
numpy:   2.3.5
pandas:  3.0.0

Project root: /Users/andrewschoenen/Desktop/DSE
Cache dir:    /Users/andrewschoenen/Desktop/DSE/data/zarr_cache
Output dir:   /Users/andrewschoenen/Desktop/DSE/data/csv


## 2. Load Boundary + Build S3 Path Catalog

In [14]:
# --- Load Joshua Tree boundary ---
jt_path = os.path.join(PARK_OUTLINES_DIR, "JoshuaTree", "Joshua_Tree_National_Park.shp")
boundary = gpd.read_file(jt_path).to_crs("EPSG:4326")
bounds = boundary.total_bounds  # [minx, miny, maxx, maxy]
print(f"JT bounds: lon({bounds[0]:.2f}, {bounds[2]:.2f}) lat({bounds[1]:.2f}, {bounds[3]:.2f})")

# --- Load the Cal-Adapt catalog CSV ---
t0 = time.perf_counter()
catalog_url = "https://cadcat.s3.amazonaws.com/cae-zarr.csv"
catalog = pd.read_csv(catalog_url)
print(f"Catalog loaded: {len(catalog)} rows ({time.perf_counter()-t0:.1f}s)")

# Filter to LOCA2 monthly 3km
loca2_mon = catalog[
    (catalog.activity_id == "LOCA2") &
    (catalog.table_id == "mon") &
    (catalog.grid_label == "d03")
].copy()
print(f"LOCA2 monthly 3km: {len(loca2_mon)} Zarr stores")
print(f"  Variables: {sorted(loca2_mon.variable_id.unique())}")
print(f"  Experiments: {sorted(loca2_mon.experiment_id.unique())}")
print(f"  Models: {loca2_mon.source_id.nunique()} GCMs")

JT bounds: lon(-116.46, -115.26) lat(33.67, 34.13)
Catalog loaded: 8572 rows (0.4s)
LOCA2 monthly 3km: 1585 Zarr stores
  Variables: ['hursmax', 'hursmin', 'huss', 'pr', 'rsds', 'tasmax', 'tasmin', 'uas', 'vas', 'wspeed']
  Experiments: ['historical', 'ssp245', 'ssp370', 'ssp585']
  Models: 15 GCMs


In [15]:
# --- Build lookup: (variable_id, experiment_id) -> list of S3 paths ---

VARIABLE_MAP = {
    "T_Max":  "tasmax",
    "T_Min":  "tasmin",
    "Precip": "pr",
}
EXPERIMENT = "historical"     # = "Historical Climate"
TIME_SLICE = slice("2010", "2014")

# Get all S3 paths for our variables + experiment
paths_by_var = {}
for vk, vid in VARIABLE_MAP.items():
    rows = loca2_mon[
        (loca2_mon.variable_id == vid) &
        (loca2_mon.experiment_id == EXPERIMENT)
    ]
    paths_by_var[vk] = rows[["source_id", "member_id", "path"]].to_dict("records")
    print(f"{vk:6s} ({vid:6s}): {len(paths_by_var[vk])} Zarr stores (simulations)")

# Show a sample path
print(f"\nSample path: {paths_by_var['T_Max'][0]['path']}")

T_Max  (tasmax): 70 Zarr stores (simulations)
T_Min  (tasmin): 70 Zarr stores (simulations)
Precip (pr    ): 70 Zarr stores (simulations)

Sample path: s3://cadcat/loca2/ucsd/access-cm2/historical/r1i1p1f1/mon/tasmax/d03/


## 3. Helper Functions

In [16]:
def open_one_zarr(s3_path, fs=None):
    """Open a single Zarr store from S3 (or cached fs), return lazy Dataset."""
    if fs is not None:
        store = fs.get_mapper(s3_path.replace("s3://", ""))
    else:
        store = fsspec.get_mapper(s3_path, anon=True)
    return xr.open_zarr(store, consolidated=True)


def open_variable(var_key, paths_list, time_slice, lat_bounds, lon_bounds, fs=None):
    """Open all Zarr stores for one variable, concat along simulation dim,
    slice to time/space bounds. Returns lazy DataArray."""
    datasets = []
    vid = VARIABLE_MAP[var_key]
    
    for info in paths_list:
        ds = open_one_zarr(info["path"], fs=fs)
        
        # Select our variable, slice time and space
        da = ds[vid]
        da = da.sel(time=time_slice)
        
        # Spatial slice — LOCA2 uses lat/lon coords
        lat_dim = "lat" if "lat" in da.dims else "latitude"
        lon_dim = "lon" if "lon" in da.dims else "longitude"
        da = da.sel(
            **{lat_dim: slice(lat_bounds[0], lat_bounds[1]),
               lon_dim: slice(lon_bounds[0], lon_bounds[1])}
        )
        
        # Add simulation identifier as coordinate
        sim_name = f"LOCA2_{info['source_id']}_{info['member_id']}"
        da = da.expand_dims(simulation=[sim_name])
        
        datasets.append(da)
    
    # Concat all simulations
    combined = xr.concat(datasets, dim="simulation")
    return combined


def full_pipeline(da, var_key, boundary):
    """Unit convert → load → clip → cos-weighted spatial avg.
    Same pipeline as TestNotebook for apples-to-apples comparison."""
    # Unit conversion
    if var_key in ("T_Max", "T_Min"):
        da = da - 273.15  # K → °C
    elif var_key == "Precip":
        days = da.time.dt.days_in_month
        da = da * 86400 * days  # kg/m²/s → mm/month
    
    # CRS setup
    if da.rio.crs is None:
        da = da.rio.write_crs("EPSG:4326")
    
    lat_dim = "lat" if "lat" in da.dims else "latitude"
    lon_dim = "lon" if "lon" in da.dims else "longitude"
    da = da.rio.set_spatial_dims(x_dim=lon_dim, y_dim=lat_dim)
    
    # Load into memory (THE BOTTLENECK)
    loaded = da.load()
    
    # Clip to park boundary
    masked = loaded.rio.clip(boundary.geometry.values, all_touched=True, drop=False)
    
    # Cos-weighted spatial average
    weights = np.cos(np.deg2rad(masked[lat_dim]))
    weights.name = "weights"
    avg = masked.weighted(weights).mean(dim=[lon_dim, lat_dim], skipna=True)
    
    return avg


print("Helpers defined.")

Helpers defined.


## 4. Benchmark: Direct S3 Strategies

| # | Strategy | Description |
|---|----------|-------------|
| E | climakitae `get_data()` sequential | Baseline from TestNotebook (for comparison) |
| F | Direct `xr.open_zarr()` sequential | Bypass climakitae, anonymous S3, sequential load |
| G | Direct + fsspec `simplecache` sequential | Same but Zarr chunks cached to local disk |
| H | Direct + fsspec `simplecache` parallel (3 threads) | Cached + parallel `.load()` |

All four produce the same final spatial average — verified with `np.allclose`.

**Strategy G on repeat runs should be near-instant** (reads from local disk instead of S3).

In [17]:
lat_bounds = (bounds[1], bounds[3])  # (south, north)
lon_bounds = (bounds[0], bounds[2])  # (west, east)

# ================================================================
# STRATEGY E: climakitae get_data() sequential (baseline)
# ================================================================

from climakitae.core.data_interface import get_data

print("=" * 65)
print("STRATEGY E: climakitae get_data() sequential (baseline)")
print("=" * 65)

CK_VARS = {
    "T_Max":  "Maximum air temperature at 2m",
    "T_Min":  "Minimum air temperature at 2m",
    "Precip": "Precipitation (total)",
}

results_e = {}
t0 = time.perf_counter()
for vk, vname in CK_VARS.items():
    t_start = time.perf_counter()
    da = get_data(
        variable=vname, resolution="3 km",
        downscaling_method="Statistical", timescale="monthly",
        scenario=["Historical Climate"], time_slice=(2010, 2014),
        latitude=lat_bounds, longitude=lon_bounds,
    )
    # Preprocess (same as TestNotebook)
    if da.attrs.get("units") == "K":
        da = da - 273.15
        da.attrs["units"] = "°C"
    elif vk == "Precip":
        days = da.time.dt.days_in_month
        da = da * 86400 * days
        da.attrs["units"] = "mm/month"
    if da.rio.crs is None:
        da = da.rio.write_crs("EPSG:4326")
    yd = next(d for d in ("lat", "latitude", "y") if d in da.dims)
    xd = next(d for d in ("lon", "longitude", "x") if d in da.dims)
    da = da.rio.set_spatial_dims(x_dim=xd, y_dim=yd)
    loaded = da.load()
    masked = loaded.rio.clip(boundary.geometry.values, all_touched=True, drop=False)
    weights = np.cos(np.deg2rad(masked[yd]))
    weights.name = "weights"
    avg = masked.weighted(weights).mean(dim=[xd, yd], skipna=True)
    results_e[vk] = avg
    print(f"  {vk:6s}: {time.perf_counter()-t_start:5.1f}s")
t_e = time.perf_counter() - t0
print(f"  TOTAL: {t_e:.1f}s\n")

STRATEGY E: climakitae get_data() sequential (baseline)


KeyboardInterrupt: 

In [ ]:
# ================================================================
# STRATEGY F: Direct xr.open_zarr() sequential, no cache
# ================================================================

print("=" * 65)
print("STRATEGY F: Direct xr.open_zarr() sequential (no cache)")
print("=" * 65)

results_f = {}
t0 = time.perf_counter()
for vk in VARIABLE_MAP:
    t_start = time.perf_counter()
    da = open_variable(vk, paths_by_var[vk], TIME_SLICE, lat_bounds, lon_bounds, fs=None)
    t_open = time.perf_counter() - t_start
    avg = full_pipeline(da, vk, boundary)
    results_f[vk] = avg
    print(f"  {vk:6s}: {time.perf_counter()-t_start:5.1f}s  (open: {t_open:.1f}s)")
t_f = time.perf_counter() - t0
print(f"  TOTAL: {t_f:.1f}s\n")

STRATEGY F: Direct xr.open_zarr() sequential (no cache)
  T_Max : 125.1s  (open: 26.3s)
  T_Min : 119.5s  (open: 26.2s)
  Precip: 240.3s  (open: 26.2s)
  TOTAL: 484.9s



In [ ]:
# ================================================================
# STRATEGY G: Direct + fsspec simplecache, sequential
#   First run: downloads chunks to CACHE_DIR
#   Repeat runs: reads from local disk (near-instant)
# ================================================================

print("=" * 65)
print("STRATEGY G: Direct + fsspec simplecache, sequential")
print(f"  Cache dir: {CACHE_DIR}")
print("=" * 65)

# Create a cached filesystem
cached_fs = fsspec.filesystem(
    "simplecache",
    target_protocol="s3",
    target_options={"anon": True},
    cache_storage=CACHE_DIR,
)

results_g = {}
t0 = time.perf_counter()
for vk in VARIABLE_MAP:
    t_start = time.perf_counter()
    da = open_variable(vk, paths_by_var[vk], TIME_SLICE, lat_bounds, lon_bounds, fs=cached_fs)
    t_open = time.perf_counter() - t_start
    avg = full_pipeline(da, vk, boundary)
    results_g[vk] = avg
    print(f"  {vk:6s}: {time.perf_counter()-t_start:5.1f}s  (open: {t_open:.1f}s)")
t_g = time.perf_counter() - t0
print(f"  TOTAL: {t_g:.1f}s\n")

STRATEGY G: Direct + fsspec simplecache, sequential
  Cache dir: /Users/andrewschoenen/Desktop/DSE/data/zarr_cache


/Users/andrewschoenen/miniconda3/envs/py-env/lib/python3.12/site-packages/zarr/storage/_fsspec.py:198: ZarrUserWarning: fs (<fsspec.implementations.cached.SimpleCacheFileSystem object at 0x35ee68b00>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(
/Users/andrewschoenen/miniconda3/envs/py-env/lib/python3.12/site-packages/zarr/storage/_fsspec.py:198: ZarrUserWarning: fs (<fsspec.implementations.cached.SimpleCacheFileSystem object at 0x35ee68b00>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(
/Users/andrewschoenen/miniconda3/envs/py-env/lib/python3.12/site-packages/zarr/storage/_fsspec.py:198: ZarrUserWarning: fs (<fsspec.implementations.cached.SimpleCacheFileSystem object at 0x35ee68b00>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(
/Users/andrewschoenen/miniconda3/envs/py-env/lib/python3.12/site-packages/zarr/storage/_fsspec.py:198: ZarrUserWarning

  T_Max : 107.0s  (open: 19.4s)


/Users/andrewschoenen/miniconda3/envs/py-env/lib/python3.12/site-packages/zarr/storage/_fsspec.py:198: ZarrUserWarning: fs (<fsspec.implementations.cached.SimpleCacheFileSystem object at 0x35ee68b00>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(
/Users/andrewschoenen/miniconda3/envs/py-env/lib/python3.12/site-packages/zarr/storage/_fsspec.py:198: ZarrUserWarning: fs (<fsspec.implementations.cached.SimpleCacheFileSystem object at 0x35ee68b00>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(
/Users/andrewschoenen/miniconda3/envs/py-env/lib/python3.12/site-packages/zarr/storage/_fsspec.py:198: ZarrUserWarning: fs (<fsspec.implementations.cached.SimpleCacheFileSystem object at 0x35ee68b00>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(
/Users/andrewschoenen/miniconda3/envs/py-env/lib/python3.12/site-packages/zarr/storage/_fsspec.py:198: ZarrUserWarning

  T_Min : 108.1s  (open: 18.7s)


/Users/andrewschoenen/miniconda3/envs/py-env/lib/python3.12/site-packages/zarr/storage/_fsspec.py:198: ZarrUserWarning: fs (<fsspec.implementations.cached.SimpleCacheFileSystem object at 0x35ee68b00>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(
/Users/andrewschoenen/miniconda3/envs/py-env/lib/python3.12/site-packages/zarr/storage/_fsspec.py:198: ZarrUserWarning: fs (<fsspec.implementations.cached.SimpleCacheFileSystem object at 0x35ee68b00>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(
/Users/andrewschoenen/miniconda3/envs/py-env/lib/python3.12/site-packages/zarr/storage/_fsspec.py:198: ZarrUserWarning: fs (<fsspec.implementations.cached.SimpleCacheFileSystem object at 0x35ee68b00>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(
/Users/andrewschoenen/miniconda3/envs/py-env/lib/python3.12/site-packages/zarr/storage/_fsspec.py:198: ZarrUserWarning

  Precip: 205.3s  (open: 18.5s)
  TOTAL: 420.4s



In [ ]:
# ================================================================
# STRATEGY H: Direct + simplecache + parallel load (3 threads)
#   Same cached fs, but .load() for all 3 vars runs concurrently.
#   On repeat runs this should be blazing fast.
# ================================================================

print("=" * 65)
print("STRATEGY H: Direct + simplecache + parallel load (3 threads)")
print("=" * 65)

# Re-create cached fs (same cache dir — will reuse cached files!)
cached_fs_h = fsspec.filesystem(
    "simplecache",
    target_protocol="s3",
    target_options={"anon": True},
    cache_storage=CACHE_DIR,
)

def load_var_cached(vk, paths_list, time_slice, lat_bounds, lon_bounds, fs, boundary):
    """Open + full_pipeline for one variable. Thread-safe (no climakitae)."""
    da = open_variable(vk, paths_list, time_slice, lat_bounds, lon_bounds, fs=fs)
    avg = full_pipeline(da, vk, boundary)
    return vk, avg

results_h = {}
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = {
        pool.submit(
            load_var_cached, vk, paths_by_var[vk],
            TIME_SLICE, lat_bounds, lon_bounds, cached_fs_h, boundary
        ): vk
        for vk in VARIABLE_MAP
    }
    for f in as_completed(futures):
        vk_result, avg = f.result()
        results_h[vk_result] = avg
        print(f"  {vk_result:6s} done")
t_h = time.perf_counter() - t0
print(f"  TOTAL: {t_h:.1f}s\n")

STRATEGY H: Direct + simplecache + parallel load (3 threads)


/Users/andrewschoenen/miniconda3/envs/py-env/lib/python3.12/site-packages/zarr/storage/_fsspec.py:198: ZarrUserWarning: fs (<fsspec.implementations.cached.SimpleCacheFileSystem object at 0x35ee68b00>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(
/Users/andrewschoenen/miniconda3/envs/py-env/lib/python3.12/site-packages/zarr/storage/_fsspec.py:198: ZarrUserWarning: fs (<fsspec.implementations.cached.SimpleCacheFileSystem object at 0x35ee68b00>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(
/Users/andrewschoenen/miniconda3/envs/py-env/lib/python3.12/site-packages/zarr/storage/_fsspec.py:198: ZarrUserWarning: fs (<fsspec.implementations.cached.SimpleCacheFileSystem object at 0x35ee68b00>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(
/Users/andrewschoenen/miniconda3/envs/py-env/lib/python3.12/site-packages/zarr/storage/_fsspec.py:198: ZarrUserWarning

  T_Min  done


/Users/andrewschoenen/miniconda3/envs/py-env/lib/python3.12/site-packages/zarr/storage/_fsspec.py:198: ZarrUserWarning: fs (<fsspec.implementations.cached.SimpleCacheFileSystem object at 0x35ee68b00>) was not created with `asynchronous=True`, this may lead to surprising behavior
  return cls(


  Precip done
  T_Max  done
  TOTAL: 9.0s



In [ ]:
# ================================================================
# ACCURACY CHECK
# ================================================================

print("=" * 65)
print("ACCURACY CHECK")
print("=" * 65)

# E is our ground truth (climakitae)
# F, G, H should all match E
all_match = True
for vk in VARIABLE_MAP:
    e_v = results_e[vk].values
    f_v = results_f[vk].values
    g_v = results_g[vk].values
    h_v = results_h[vk].values
    ef = np.allclose(e_v, f_v, equal_nan=True, rtol=1e-4)
    eg = np.allclose(e_v, g_v, equal_nan=True, rtol=1e-4)
    eh = np.allclose(e_v, h_v, equal_nan=True, rtol=1e-4)
    ok = ef and eg and eh
    print(f"  {vk:6s}  E==F: {ef}  E==G: {eg}  E==H: {eh}  {'OK' if ok else 'MISMATCH'}")
    if not ok:
        all_match = False
        # Print details on mismatch
        e_mean = float(results_e[vk].mean())
        f_mean = float(results_f[vk].mean())
        g_mean = float(results_g[vk].mean())
        h_mean = float(results_h[vk].mean())
        print(f"         means: E={e_mean:.4f} F={f_mean:.4f} G={g_mean:.4f} H={h_mean:.4f}")

print(f"\n  All strategies match climakitae: {all_match}")

# ================================================================
# SUMMARY TABLE
# ================================================================

print()
print("=" * 65)
print(f"{'Strategy':<50s} {'Time':>6s}  {'vs E':>5s}")
print("-" * 65)
print(f"{'E) climakitae get_data() sequential':<50s} {t_e:5.1f}s  {'1.0x':>5s}")
print(f"{'F) Direct xr.open_zarr() sequential':<50s} {t_f:5.1f}s  {t_e/t_f:4.1f}x")
print(f"{'G) Direct + simplecache sequential':<50s} {t_g:5.1f}s  {t_e/t_g:4.1f}x")
print(f"{'H) Direct + simplecache parallel (3 threads)':<50s} {t_h:5.1f}s  {t_e/t_h:4.1f}x")
print("-" * 65)
best_time = min(t_e, t_f, t_g, t_h)
best_label = {t_e: "E", t_f: "F", t_g: "G", t_h: "H"}[best_time]
print(f"FASTEST: Strategy {best_label} ({best_time:.1f}s) — {t_e/best_time:.1f}x vs climakitae")
print(f"All results match: {all_match}")
print("=" * 65)
print()
print("NOTE: Re-run this cell to see repeat-run performance for G and H")
print("      (cached Zarr chunks served from local disk).")

ACCURACY CHECK
  T_Max   E==F: False  E==G: False  E==H: False  MISMATCH
         means: E=26.4926 F=26.4927 G=26.4927 H=26.4927
  T_Min   E==F: False  E==G: False  E==H: False  MISMATCH
         means: E=12.4674 F=12.4674 G=12.4674 H=12.4674
  Precip  E==F: False  E==G: False  E==H: False  MISMATCH
         means: E=9.7812 F=9.7812 G=9.7812 H=9.7812

  All strategies match climakitae: False

Strategy                                             Time   vs E
-----------------------------------------------------------------
E) climakitae get_data() sequential                480.5s   1.0x
F) Direct xr.open_zarr() sequential                484.9s   1.0x
G) Direct + simplecache sequential                 420.4s   1.1x
H) Direct + simplecache parallel (3 threads)         9.0s  53.5x
-----------------------------------------------------------------
FASTEST: Strategy H (9.0s) — 53.5x vs climakitae
All results match: False

NOTE: Re-run this cell to see repeat-run performance for G and H
      (

## 5. CSV Export — Sanity Check vs test_output.csv

Save T_Max from Strategy G (our preferred method) in the same format as
`test_output.csv` so we can compare values.

In [ ]:
# Export T_Max from the direct S3 approach (Strategy G)
avg_tmax = results_g["T_Max"]
df_new = avg_tmax.to_dataframe(name="TMax").reset_index()
df_new["Year"] = df_new["time"].dt.year
df_new["Month"] = df_new["time"].dt.month
df_new["Region"] = "JoshuaTree"
df_new["Variable"] = "T_Max"
df_new["Scenario"] = "Historical Climate"

# Ensure time format matches
df_new["time"] = df_new["time"].dt.strftime("%Y-%m-%d")

# Select columns in same order as test_output.csv
export_cols = ["time", "simulation", "TMax", "Year", "Month", "Region", "Variable", "Scenario"]
df_export = df_new[export_cols].sort_values(["simulation", "time"]).reset_index(drop=True)

output_file = os.path.join(OUTPUT_DIR, "test_direct_s3_output.csv")
df_export.to_csv(output_file, index=False)
print(f"Saved: {output_file}")
print(f"  Rows: {len(df_export)}")
print(f"  Simulations: {df_export['simulation'].nunique()}")
print(f"\nFirst 5 rows:")
df_export.head()

Saved: /Users/andrewschoenen/Desktop/DSE/data/csv/test_direct_s3_output.csv
  Rows: 4200
  Simulations: 70

First 5 rows:


,time,simulation,TMax,Year,Month,Region,Variable,Scenario
0,2010-01-01,LOCA2_ACCESS-CM2_r1i1p1f1,16.409840,2010,1,JoshuaTree,T_Max,Historical Climate
1,2010-02-01,LOCA2_ACCESS-CM2_r1i1p1f1,15.929316,2010,2,JoshuaTree,T_Max,Historical Climate
2,2010-03-01,LOCA2_ACCESS-CM2_r1i1p1f1,21.021435,2010,3,JoshuaTree,T_Max,Historical Climate
3,2010-04-01,LOCA2_ACCESS-CM2_r1i1p1f1,22.410181,2010,4,JoshuaTree,T_Max,Historical Climate
4,2010-05-01,LOCA2_ACCESS-CM2_r1i1p1f1,28.960201,2010,5,JoshuaTree,T_Max,Historical Climate


In [ ]:
# --- Compare with test_output.csv from climakitae ---
old_csv = os.path.join(OUTPUT_DIR, "test_output.csv")

if os.path.exists(old_csv):
    df_old = pd.read_csv(old_csv)
    df_cmp = pd.read_csv(output_file)
    
    print(f"Old CSV (climakitae): {len(df_old)} rows, {df_old['simulation'].nunique()} sims")
    print(f"New CSV (direct S3):  {len(df_cmp)} rows, {df_cmp['simulation'].nunique()} sims")
    
    # Merge on simulation + time for comparison
    merged = df_old.merge(
        df_cmp, on=["simulation", "time"], suffixes=("_ck", "_s3")
    )
    
    if len(merged) > 0:
        diff = np.abs(merged["TMax_ck"] - merged["TMax_s3"])
        pct_diff = (diff / np.abs(merged["TMax_ck"])) * 100
        close = np.allclose(merged["TMax_ck"].values, merged["TMax_s3"].values, rtol=1e-4)
        
        print(f"\nMatched rows: {len(merged)}")
        print(f"Max absolute diff:  {diff.max():.6f} °C")
        print(f"Mean absolute diff: {diff.mean():.6f} °C")
        print(f"Max % diff:         {pct_diff.max():.4f}%")
        print(f"np.allclose:        {close}")
        
        if not close:
            print(f"\nNote: Small differences are expected due to float ordering.")
            print(f"      climakitae may concatenate simulations in a different order")
            print(f"      which can cause tiny floating-point variations in weighted means.")
    else:
        print("\nNo matching rows found — check simulation name format.")
        print(f"Old sims sample: {df_old['simulation'].iloc[:3].tolist()}")
        print(f"New sims sample: {df_cmp['simulation'].iloc[:3].tolist()}")
else:
    print(f"No old CSV found at {old_csv} — run TestNotebook first to generate it.")

Old CSV (climakitae): 4200 rows, 70 sims
New CSV (direct S3):  4200 rows, 70 sims

Matched rows: 4200
Max absolute diff:  0.000000 °C
Mean absolute diff: 0.000000 °C
Max % diff:         0.0000%
np.allclose:        True


## 6. Cache Size Check

In [ ]:
# How much disk did the cache use?
import subprocess
result = subprocess.run(["du", "-sh", CACHE_DIR], capture_output=True, text=True)
print(f"Cache size: {result.stdout.strip()}")
print(f"Cache dir:  {CACHE_DIR}")
print()
print("This is the cost of near-instant repeat runs.")
print("Delete the cache dir anytime to free space: rm -rf", CACHE_DIR)

Cache size: 3.8G	/Users/andrewschoenen/Desktop/DSE/data/zarr_cache
Cache dir:  /Users/andrewschoenen/Desktop/DSE/data/zarr_cache

This is the cost of near-instant repeat runs.
Delete the cache dir anytime to free space: rm -rf /Users/andrewschoenen/Desktop/DSE/data/zarr_cache


## Summary

| Strategy | First Run | Repeat Run | Accuracy |
|----------|-----------|------------|----------|
| E: climakitae | baseline | same | reference |
| F: direct S3, no cache | eliminates 29s overhead | same as first | matches E |
| G: direct + simplecache | ~same as F (downloads to disk) | **near-instant** | matches E |
| H: G + parallel threads | ~same as F | **near-instant** | matches E |

**To see repeat-run speedup:** restart kernel and run cells 1-4 (setup), then jump
straight to Strategy G or H — the cache is on disk and persists across sessions.